# DoRA in Diffusion Models: Traditional Chinese Landscape Paintings


## 1. Environment Setup

In [ ]:
!pip install -q diffusers transformers accelerate datasets

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

In [ ]:
import os
import gc
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from typing import Iterable, Tuple

from diffusers import StableDiffusionXLPipeline, DDPMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import AutoTokenizer, PretrainedConfig, CLIPTextModel, CLIPTextModelWithProjection

print("Environment setup complete. All libraries imported successfully.")

## 2. Configuration

Setting up configuration for diffusion model, including base model ID, dataset path, instance prompt for training, DoRA hyperparameters (rank, alpha, dropout), training hyperparameters (batch size, steps, learning rate), and hardware configurations (device and weight data type).

In [ ]:
# Model and Dataset paths
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DATASET_ID = "Traditional-Chinese-Paintings"
DATA_DIR = "/content/drive/My Drive/Colab_Notebooks/DoRA/Small-set"
INSTANCE_PROMPT = "a sks traditional Chinese ink wash landscape painting"

# DoRA Hyperparameters
RANK = 8
ALPHA = 16
DROPOUT = 0.0

# Training Hyperparameters
BATCH_SIZE = 2
STEPS = 1000
LR = 1e-4

# Hardware Configurations
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Automatically use bfloat16 if supported by GPU (e.g., A100, H100, RTX 30/40 series)
WEIGHT_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

print(f"Configuration loaded.")
print(f"Running on DEVICE: {DEVICE} | Precision: {WEIGHT_DTYPE}")

## 3. LoRA / DoRA / Full-FT Layers

Define LoRA / DoRA/ Full-FT and inject into UNet of diffusion model.

For a frozen pretrained weight `W0`, DoRA trains a magnitude vector `m` and low-rank direction update `BA`:

$$W' = m \frac{W_0 + BA}{\lVert W_0 + BA \rVert_c}$$



In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float):
        super().__init__()
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False) if base.bias is not None else None

        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=self.weight.device, dtype=self.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=self.weight.device, dtype=self.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        base_out = F.linear(x, self.weight, self.bias)
        if self.training:
            lora_out = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling
        else:
            lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B) * self.scaling
        return base_out + lora_out

class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float):
        super().__init__()
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Freeze Original Weights
        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False) if base.bias is not None else None

        # LoRA Parameters
        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=self.weight.device, dtype=self.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=self.weight.device, dtype=self.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        # Magnitude Parameters
        magnitude = torch.linalg.vector_norm(self.weight.float(), ord=2, dim=1).to(self.weight.dtype)
        self.magnitude = nn.Parameter(magnitude)

    def merged_weight(self):
        """Merge weight for inference time"""
        directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
        norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1, keepdim=True).to(directional.dtype).clamp_min(1e-6)
        return self.magnitude[:, None] * directional / norm

    def forward(self, x):
        if self.training:
            directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
            norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1).to(directional.dtype).clamp_min(1e-6)
            norm_scale = self.magnitude / norm.detach()

            base_out = F.linear(x, self.weight, None)
            lora_out = F.linear(
            F.linear(self.dropout(x), self.lora_A), self.lora_B) * self.scaling

            out = norm_scale * (base_out + lora_out)

            if self.bias is not None:
                out = out + self.bias

            return out

        else:
            return F.linear(x, self.merged_weight(), self.bias)


def inject_and_unfreeze(model: nn.Module, target_modules: list, rank: int, alpha: int, dropout: float, method: str) -> tuple:
    """return (# of substituted layer, # of trainable params, # of total params)"""
    model.requires_grad_(False)

    modules = dict(model.named_modules())
    replacements = []


    for name, module in modules.items():
        if isinstance(module, nn.Linear) and any(target in name for target in target_modules):
            replacements.append(name)

    replace_count = 0
    for name in replacements:
        parent_name, child_name = name.rsplit('.', 1) if '.' in name else ('', name)
        parent = modules[parent_name] if parent_name else model
        old_layer = getattr(parent, child_name)

        if method == "dora":
            new_layer = DoRALinear(old_layer, rank=rank, alpha=alpha, dropout=dropout)
            setattr(parent, child_name, new_layer)
            # Unfreeze DoRA parameters
            new_layer.lora_A.requires_grad_(True)
            new_layer.lora_B.requires_grad_(True)
            new_layer.magnitude.requires_grad_(True)
            replace_count += 1

        elif method == "lora":
            new_layer = LoRALinear(old_layer, rank=rank, alpha=alpha, dropout=dropout)
            setattr(parent, child_name, new_layer)
            # Unfreeze LoRA parameters
            new_layer.lora_A.requires_grad_(True)
            new_layer.lora_B.requires_grad_(True)
            replace_count += 1

        elif method == "full":
            # Unfreeze complete target layer
            old_layer.weight.requires_grad_(True)
            if old_layer.bias is not None:
                old_layer.bias.requires_grad_(True)
            replace_count += 1

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return replace_count, trainable, total

print("Core modules defined.")

### Smoketest Sanity Check For DoRA

In [ ]:
print("Running DoRALinear startup identity test...")

_test_layer = nn.Linear(7, 4, bias=False)
_wrap_layer = DoRALinear(_test_layer, rank=2, alpha=4, dropout=0.0)
_dummy_input = torch.randn(3, 7)

# Verify outputs match exactly at initialization
with torch.no_grad():
    assert torch.allclose(_test_layer(_dummy_input), _wrap_layer(_dummy_input), atol=1e-5), "Mismatch detected!"

print("SUCCESS: Identity test passed. DoRA starts cleanly.")

## 4. Dataset Loading



In [ ]:
class DreamBoothDataset(Dataset):
    def __init__(self, data_dir, prompt, size=1024):

        if not os.path.exists(data_dir):
            raise FileNotFoundError(f"❌ Cannot find directory: {data_dir}，please check that Drive is mounted and the directory is correct")

        print(f"Reading datasets in {data_dir}...")


        self.dataset = load_dataset(
            "imagefolder",
            data_dir=data_dir,
            split="train",
            drop_labels=True
        )
        self.prompt = prompt

        self.transform = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.RandomCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image = self.dataset[index]["image"]
        if not image.mode == "RGB":
            image = image.convert("RGB")
        return {"pixel_values": self.transform(image), "prompt": self.prompt}

# Test load
_dummy_dataset = DreamBoothDataset(DATA_DIR, INSTANCE_PROMPT)
print(f"✅ Dataset successfully loaded. Total samples: {_dummy_dataset.__len__()}")

## 5. Smoketests


### Test 1: Runtime / dtype / Cache Sanity Check

In [ ]:
import os
import gc
import torch

print("===== Runtime Sanity Check =====")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

    WEIGHT_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    WEIGHT_DTYPE = torch.float32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", DEVICE)
print("WEIGHT_DTYPE:", WEIGHT_DTYPE)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Allocated VRAM:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
    print("Reserved VRAM:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

### Test 2: Cache Cleanup and Forced Pre-download

In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

# 1. Clean up potential Hugging Face download locks to prevent 0% stuck issue
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
print("Checking for corrupted Hugging Face cache locks...")
if os.path.exists(cache_dir):
    lock_count = 0
    for root, dirs, files in os.walk(cache_dir):
        for d in dirs:
            if d == ".locks":
                lock_path = os.path.join(root, d)
                print(f"Removing lock directory: {lock_path}")
                shutil.rmtree(lock_path)
                lock_count += 1
    print(f"Cleanup finished. Removed {lock_count} lock folders.")
else:
    print("No cache directory found, skipping cleanup.")

# 2. Force multi-threaded download of essential SDXL components
print(f"Initiating pre-download for {MODEL_ID}. Please wait...")
try:
    snapshot_download(
        repo_id=MODEL_ID,
        resume_download=True,
        max_workers=4,
        allow_patterns=["*.json", "*.fp16.safetensors", "*.safetensors", "*.txt", "*model.safetensors"],
        ignore_patterns=["*.msgpack", "*.bin", "*non_ema*", "*flax*"]
    )
    print("SUCCESS: Base model components are ready offline!")
except Exception as e:
    print(f"WARNING: Network fluctuation during download, but it might be complete: {e}")

### Test 3: DoRA Logic Consistency

In [ ]:
torch.manual_seed(0)

base = nn.Linear(7, 4, bias=True).to(DEVICE, dtype=WEIGHT_DTYPE)
dora = DoRALinear(base, rank=2, alpha=4, dropout=0.0).to(DEVICE)

x = torch.randn(3, 7, device=DEVICE, dtype=WEIGHT_DTYPE)

# 1. Initialization identity test
base.eval()
dora.eval()

with torch.no_grad():
    y_base = base(x)
    y_dora = dora(x)

max_diff = (y_base.float() - y_dora.float()).abs().max().item()
print("Init max diff:", max_diff)
assert max_diff < 5e-3, "DoRA init is not equivalent to base Linear."

# 2. train/eval consistency with dropout=0
dora.train()
with torch.no_grad():
    y_train = dora(x)

dora.eval()
with torch.no_grad():
    y_eval = dora(x)

max_diff_train_eval = (y_train.float() - y_eval.float()).abs().max().item()
print("Train/Eval max diff with dropout=0:", max_diff_train_eval)
assert max_diff_train_eval < 5e-3, "Train/eval outputs mismatch when dropout=0."

# 3. merged_weight consistency
with torch.no_grad():
    y_merged = F.linear(x, dora.merged_weight(), dora.bias)

max_diff_merged = (y_eval.float() - y_merged.float()).abs().max().item()
print("Eval/Merged max diff:", max_diff_merged)
assert max_diff_merged < 5e-3, "merged_weight output mismatch."

# 4. backward test
dora.train()
loss = dora(x).float().pow(2).mean()
loss.backward()

assert dora.lora_A.grad is not None, "lora_A has no grad."
assert dora.lora_B.grad is not None, "lora_B has no grad."
assert dora.magnitude.grad is not None, "magnitude has no grad."
assert dora.weight.grad is None, "Frozen base weight unexpectedly has grad."

print("✅ DoRALinear smoke test passed.")

### Test 4: Rapid Data and DoRA Injection Verification

In [ ]:
import torch
import gc

print("Testing Dataset pipeline...")
try:
    # Test loading with the existing DATA_DIR and INSTANCE_PROMPT
    _smoke_dataset = DreamBoothDataset(DATA_DIR, INSTANCE_PROMPT)
    print(f"SUCCESS: Dataset loaded. Found {len(_smoke_dataset)} images.")
    print(f"Sample Image Shape: {_smoke_dataset[0]['pixel_values'].shape}")
except Exception as e:
    print(f"FAILURE: Dataset error: {e}")

print("\nTesting DoRA injection logic...")
try:
    # Load a temporary UNet shell to test the injection mechanism
    _tmp_unet = UNet2DConditionModel.from_pretrained(
        MODEL_ID,
        subfolder="unet",
        torch_dtype=WEIGHT_DTYPE,
        local_files_only=True
    ).to(DEVICE)

    target_layers = ["to_q", "to_k", "to_v", "to_out.0"]
    repl_count, train_p, total_p = inject_and_unfreeze(_tmp_unet, target_layers, RANK, ALPHA, DROPOUT, "dora")

    print(f"SUCCESS: DoRA injected into {repl_count} layers.")
    print(f"Trainable parameters in test: {train_p / 1e6:.2f}M")

    # Critical: Clear memory immediately
    del _tmp_unet
    torch.cuda.empty_cache()
    gc.collect()
    print("Memory cleared after test.")
except Exception as e:
    print(f"FAILURE: Injection failed: {e}")

### Test 5: Single Image Inference

In [ ]:
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load a tiny test UNet from local cache
_smoke_unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID,
    subfolder="unet",
    torch_dtype=WEIGHT_DTYPE,
    local_files_only=True,
    use_safetensors=True,
).to(DEVICE)

# Optional: inject DoRA just to verify adapted UNet works in pipeline
inject_and_unfreeze(
    _smoke_unet,
    ["to_q", "to_k", "to_v", "to_out.0"],
    RANK,
    ALPHA,
    DROPOUT,
    "dora",
)

_smoke_unet.eval()

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    unet=_smoke_unet,
    torch_dtype=WEIGHT_DTYPE,
    local_files_only=True,
    use_safetensors=True,
)

if torch.cuda.is_available():
    try:
        pipe.enable_model_cpu_offload()
        print("Using CPU offload.")
    except Exception as e:
        print("CPU offload failed, using pipe.to(DEVICE):", e)
        pipe = pipe.to(DEVICE)
else:
    pipe = pipe.to(DEVICE)

try:
    pipe.enable_attention_slicing()
except Exception:
    pass

pipe.set_progress_bar_config(disable=False)

prompt = "a sks traditional Chinese ink wash landscape painting of mountains and a river"
generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(42)

with torch.inference_mode():
    img = pipe(
        prompt=prompt,
        num_inference_steps=5,
        guidance_scale=5.0,
        generator=generator,
    ).images[0]

display(img)

img.save("smoke_test_inference.png")
print("✅ Single image inference smoke test passed. Saved as smoke_test_inference.png")

del pipe, _smoke_unet
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Test 6: One-step SDXL UNet Backward

In [ ]:
_tmp_unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID,
    subfolder="unet",
    torch_dtype=WEIGHT_DTYPE,
    local_files_only=True,
    use_safetensors=True,
).to(DEVICE)

repl_count, train_p, total_p = inject_and_unfreeze(
    _tmp_unet,
    ["to_q", "to_k", "to_v", "to_out.0"],
    RANK,
    ALPHA,
    DROPOUT,
    "dora",
)

print("DoRA layers:", repl_count)
print("Trainable params:", train_p)

_tmp_unet.train()
optimizer = torch.optim.AdamW(
    [p for p in _tmp_unet.parameters() if p.requires_grad],
    lr=1e-4,
)

# SDXL latent shape for 1024x1024 image is 4 x 128 x 128
dummy_latents = torch.randn(1, 4, 128, 128, device=DEVICE, dtype=WEIGHT_DTYPE)
dummy_noise = torch.randn_like(dummy_latents)
dummy_t = torch.tensor([500], device=DEVICE, dtype=torch.long)

# SDXL prompt_embeds dim = 2048
dummy_prompt_embeds = torch.randn(1, 77, 2048, device=DEVICE, dtype=WEIGHT_DTYPE)

dummy_added_cond_kwargs = {
    "text_embeds": torch.randn(1, 1280, device=DEVICE, dtype=WEIGHT_DTYPE),
    "time_ids": torch.tensor(
        [[1024, 1024, 0, 0, 1024, 1024]],
        device=DEVICE,
        dtype=WEIGHT_DTYPE,
    ),
}

optimizer.zero_grad(set_to_none=True)

pred = _tmp_unet(
    dummy_latents,
    dummy_t,
    dummy_prompt_embeds,
    added_cond_kwargs=dummy_added_cond_kwargs,
).sample

loss = F.mse_loss(pred.float(), dummy_noise.float())
print("Loss:", loss.item())

loss.backward()
optimizer.step()

print("✅ One-step backward passed.")

del _tmp_unet, optimizer, dummy_latents, dummy_noise, dummy_prompt_embeds, dummy_added_cond_kwargs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Test 7: Mini Pipeline

In [ ]:
TEST_STEPS = 5
TEST_METHODS = ["lora", "dora"]

print(f"Starting micro-pipeline test for methods: {TEST_METHODS}")

for method in TEST_METHODS:
    print(f"\n--- Testing Method: {method.upper()} ---")

    # 1. Setup minimal components
    unet = UNet2DConditionModel.from_pretrained(
        MODEL_ID, subfolder="unet", torch_dtype=WEIGHT_DTYPE, local_files_only=True
    ).to(DEVICE)

    # 2. Apply method
    inject_and_unfreeze(unet, ["to_q", "to_k", "to_v", "to_out.0"], RANK, ALPHA, DROPOUT, method)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, unet.parameters()), lr=1e-4)

    # 3. Create dummy batch for backprop test
    unet.train()
    dummy_latents = torch.randn(1, 4, 128, 128, device=DEVICE, dtype=WEIGHT_DTYPE)
    dummy_noise = torch.randn_like(dummy_latents)
    dummy_t = torch.tensor([500], device=DEVICE, dtype=torch.long)
    dummy_embeds = torch.randn(1, 77, 2048, device=DEVICE, dtype=WEIGHT_DTYPE)
    dummy_kwargs = {
        "text_embeds": torch.randn(1, 1280, device=DEVICE, dtype=WEIGHT_DTYPE),
        "time_ids": torch.tensor([[1024, 1024, 0, 0, 1024, 1024]], dtype=WEIGHT_DTYPE, device=DEVICE)
    }

    # 4. Step through mini-loop
    for i in range(TEST_STEPS):
        optimizer.zero_grad()
        pred = unet(dummy_latents, dummy_t, dummy_embeds, added_cond_kwargs=dummy_kwargs).sample
        loss = F.mse_loss(pred.float(), dummy_noise.float(), reduction="mean")
        loss.backward()
        optimizer.step()
        print(f"  Step {i+1}/{TEST_STEPS} | Loss: {loss.item():.4f}")

    # 5. Cleanup
    del unet, optimizer
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Method {method.upper()} passed VRAM test.")

print("\n🎉 SMOKE TEST COMPLETE: Your environment and code logic are robust.")

## 5. Pipeline of SDXL Training, Evaluation, and Visualization for Multiple Models

In [ ]:
import gc
import numpy as np
import matplotlib.pyplot as plt
from diffusers import StableDiffusionXLPipeline, DDPMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import AutoTokenizer, CLIPTextModel, CLIPTextModelWithProjection


METHODS = ["base", "lora", "dora", "full"]


experiment_results = {m: {"images": [], "losses": []} for m in METHODS}

test_prompts = [
    "a sks traditional Chinese ink wash landscape painting of rugged mountain cliffs, a secluded village, pine forests, streams, calligraphy, and red seals on aged paper",
    "a sks traditional Chinese ink wash landscape painting of a riverside hut beneath pine, willow, and bamboo, with a calm lake and distant misty mountains",
    "a sks soft traditional Chinese ink wash landscape painting of waterfowl on a placid lake, misty mountains, willow trees, and a lone fishing boat",
    "a sks formal traditional Chinese ink wash painting of jagged mountains, foreground trees, open water, calligraphy, and a large red seal on aged paper",
    "a sks traditional Chinese ink wash landscape painting of a futuristic cyberpunk city, with towering layered buildings, glowing lantern-like signs, narrow bridges, misty streets",
    "a sks traditional Chinese ink wash landscape painting of a modern sports car, sleek and streamlined, positioned in an open landscape with soft mountains and sparse trees"
]

def run_single_experiment(method: str):
    print(f"\n" + "="*50)
    print(f"STARTING EXPERIMENT: {method.upper()}")
    print("="*50)

    tokenizer_one = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer", use_fast=False)
    tokenizer_two = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer_2", use_fast=False)
    text_encoder_one = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    text_encoder_two = CLIPTextModelWithProjection.from_pretrained(MODEL_ID, subfolder="text_encoder_2", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

    vae.requires_grad_(False)
    text_encoder_one.requires_grad_(False)
    text_encoder_two.requires_grad_(False)

    losses = []

    if method != "base":
        target_modules = ["to_q", "to_k", "to_v", "to_out.0"]
        replace_count, train_p, total_p = inject_and_unfreeze(unet, target_modules, RANK, ALPHA, DROPOUT, method)
        print(f"[{method.upper()}] Trainable parameters: {train_p / 1e6:.2f}M ({(train_p/total_p)*100:.2f}%)")

        if method == "full":
            print(f"[{method.upper()}] Enabling gradient checkpointing to prevent OOM...")
            unet.enable_gradient_checkpointing()
            current_lr = 1e-5
        else:
            current_lr = LR


        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, unet.parameters()), lr=current_lr)

        train_dataset = DreamBoothDataset(DATA_DIR, INSTANCE_PROMPT)
        train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

        unet.train()
        global_step = 0
        pbar = tqdm(total=STEPS, desc=f"Training {method.upper()}")

        while global_step < STEPS:
            for batch in train_dataloader:
                if global_step >= STEPS: break
                optimizer.zero_grad()

                pixel_values = batch["pixel_values"].to(DEVICE, dtype=WEIGHT_DTYPE)
                with torch.no_grad():
                    latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor

                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=DEVICE).long()
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                prompts = batch["prompt"]
                with torch.no_grad():
                    inputs1 = tokenizer_one(prompts, padding="max_length", max_length=77, truncation=True, return_tensors="pt").to(DEVICE)
                    hidden1 = text_encoder_one(inputs1.input_ids, output_hidden_states=True).hidden_states[-2]

                    inputs2 = tokenizer_two(prompts, padding="max_length", max_length=77, truncation=True, return_tensors="pt").to(DEVICE)
                    embeds2 = text_encoder_two(inputs2.input_ids, output_hidden_states=True)
                    hidden2 = embeds2.hidden_states[-2]

                    prompt_embeds = torch.cat([hidden1, hidden2], dim=-1)
                    add_time_ids = torch.tensor([[1024, 1024, 0, 0, 1024, 1024]], dtype=WEIGHT_DTYPE, device=DEVICE).repeat(bsz, 1)
                    added_cond_kwargs = {"text_embeds": embeds2[0], "time_ids": add_time_ids}

                model_pred = unet(noisy_latents, timesteps, prompt_embeds, added_cond_kwargs=added_cond_kwargs).sample
                loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

                loss.backward()
                optimizer.step()
                losses.append(loss.item())

                pbar.update(1)
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})
                global_step += 1

    print(f"\n[{method.upper()}] Generating test images...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        unet=unet,
        torch_dtype=WEIGHT_DTYPE
    ).to(DEVICE)

    generated_images = []
    for row_idx, prompt in enumerate(test_prompts):
        seed = 42 + row_idx
        generator = torch.Generator(DEVICE).manual_seed(seed)
        img = pipe(prompt, num_inference_steps=30, guidance_scale=7.0, generator=generator).images[0]
        generated_images.append(img)

    print(f"[{method.upper()}] Cleaning up memory for next experiment...")
    del pipe, unet, vae, text_encoder_one, text_encoder_two, tokenizer_one, tokenizer_two, noise_scheduler
    if method != "base":
        del optimizer, train_dataloader
    torch.cuda.empty_cache()
    gc.collect()

    return losses, generated_images


for method in METHODS:
    l_history, imgs = run_single_experiment(method)
    experiment_results[method]["losses"] = l_history
    experiment_results[method]["images"] = imgs

print("\n ✅ ALL EXPERIMENTS COMPLETED SUCCESSFULLY!")

fig, axes = plt.subplots(len(test_prompts), len(METHODS), figsize=(16, 4 * len(test_prompts)))
plt.subplots_adjust(wspace=0.05, hspace=0.1)

labels = {"base": "Base (No FT)", "lora": "Traditional LoRA", "dora": "DoRA (Ours)", "full": "Full Fine-Tuning"}

for col_idx, method in enumerate(METHODS):
    for row_idx, prompt in enumerate(test_prompts):
        img = experiment_results[method]["images"][row_idx]

        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].axis("off")

        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"{labels[method]}", fontsize=14, fontweight="bold")

        if col_idx == 0:
            axes[row_idx, 0].text(-0.1, 0.5, prompt[:30]+'...\n'+prompt[30:],
                                  transform=axes[row_idx, 0].transAxes,
                                  fontsize=12, va='center', ha='right', rotation=0)

plt.tight_layout()
plt.savefig("ultimate_4way_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

def smooth_curve(scalars, weight=0.9):
    last = scalars[0]
    smoothed = []
    for point in scalars:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

plt.figure(figsize=(10, 6))
colors = {"full": "green", "lora": "blue", "dora": "red"}

for m in ["full", "lora", "dora"]:
    losses = experiment_results[m]["losses"]
    plt.plot(smooth_curve(losses), label=labels[m], alpha=0.9, color=colors[m], linewidth=2)
    plt.plot(losses, alpha=0.1, color=colors[m])

plt.title("Training Convergence: Full FT vs LoRA vs DoRA", fontsize=14)
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("MSE Loss (Smoothed)", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig("loss_comparison_3way.png", dpi=300)
plt.show()

print("✅ Results saved as '4way_comparison.png' and 'loss_comparison_3way.png'")

Create directory to save experiment results

In [ ]:
import os
import json
import pickle
import pandas as pd
import numpy as np

RESULT_DIR = "/content/drive/My Drive/Colab_Notebooks/DoRA/results"
IMAGE_DIR = os.path.join(RESULT_DIR, "generated_images")
LOSS_DIR = os.path.join(RESULT_DIR, "losses")

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(LOSS_DIR, exist_ok=True)

print("Results will be saved to:")
print(RESULT_DIR)

In [ ]:
# Save generated PIL images
for method, result in experiment_results.items():
    method_img_dir = os.path.join(IMAGE_DIR, method)
    os.makedirs(method_img_dir, exist_ok=True)

    images = result.get("images", [])

    for i, img in enumerate(images):
        save_path = os.path.join(method_img_dir, f"{method}_prompt_{i+1}.png")
        img.save(save_path)

    print(f"Saved {len(images)} images for {method} to {method_img_dir}")

In [ ]:
# Save losses per method
loss_summary = {}

for method, result in experiment_results.items():
    losses = result.get("losses", [])

    # Base has no training losses
    if len(losses) == 0:
        print(f"{method}: no losses to save.")
        continue

    df = pd.DataFrame({
        "step": list(range(1, len(losses) + 1)),
        "loss": losses,
    })

    csv_path = os.path.join(LOSS_DIR, f"{method}_losses.csv")
    json_path = os.path.join(LOSS_DIR, f"{method}_losses.json")

    df.to_csv(csv_path, index=False)

    with open(json_path, "w") as f:
        json.dump(losses, f, indent=2)

    loss_summary[method] = {
        "num_steps": len(losses),
        "final_loss": float(losses[-1]),
        "mean_loss": float(np.mean(losses)),
        "last_100_mean_loss": float(np.mean(losses[-100:])) if len(losses) >= 100 else float(np.mean(losses)),
    }

    print(f"Saved losses for {method}:")
    print(" CSV :", csv_path)
    print(" JSON:", json_path)

summary_path = os.path.join(LOSS_DIR, "loss_summary.json")
with open(summary_path, "w") as f:
    json.dump(loss_summary, f, indent=2)

print("Saved loss summary to:", summary_path)

In [ ]:
pickle_path = os.path.join(RESULT_DIR, "experiment_results.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(experiment_results, f)

print("Saved full experiment_results to:")
print(pickle_path)

In [ ]:
import shutil
import os

for filename in [
    "ultimate_4way_comparison.png",
    "loss_comparison_3way.png",
]:
    src = os.path.join("/content", filename)
    dst = os.path.join(RESULT_DIR, filename)

    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied {filename} to {dst}")
    else:
        print(f"Not found: {src}")

Inference Grid

In [ ]:
import os
import json
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

RESULT_DIR = "/content/drive/My Drive/Colab_Notebooks/DoRA/results-small-set-ChineseArt"
PLOT_DIR = os.path.join(RESULT_DIR, "plots")
IMAGE_DIR = os.path.join(RESULT_DIR, "generated_images")
METRIC_DIR = os.path.join(RESULT_DIR, "metrics")

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(METRIC_DIR, exist_ok=True)

print("Saving results to:")
print(RESULT_DIR)

print("Available methods:", list(experiment_results.keys()))

In [ ]:
METHODS_TO_PLOT = [m for m in ["base", "lora", "dora", "full"] if m in experiment_results]

labels = {
    "base": "Base",
    "lora": "LoRA",
    "dora": "DoRA",
    "full": "Target Full FT",
}

num_prompts = len(next(iter(experiment_results.values()))["images"])
num_methods = len(METHODS_TO_PLOT)

fig, axes = plt.subplots(num_prompts, num_methods, figsize=(4 * num_methods, 4 * num_prompts))

if num_prompts == 1:
    axes = np.expand_dims(axes, axis=0)
if num_methods == 1:
    axes = np.expand_dims(axes, axis=1)

for col_idx, method in enumerate(METHODS_TO_PLOT):
    images = experiment_results[method]["images"]

    for row_idx, img in enumerate(images):
        ax = axes[row_idx, col_idx]
        ax.imshow(img)
        ax.axis("off")

        if row_idx == 0:
            ax.set_title(labels.get(method, method), fontsize=14, fontweight="bold")

        if col_idx == 0 and "test_prompts" in globals():
            prompt_text = test_prompts[row_idx]
            ax.text(
                -0.08,
                0.5,
                prompt_text[:45] + ("..." if len(prompt_text) > 45 else ""),
                transform=ax.transAxes,
                fontsize=10,
                va="center",
                ha="right",
            )

plt.tight_layout()

save_path = os.path.join(PLOT_DIR, "inference_grid.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved inference grid to:", save_path)

In [ ]:
# Losses and Plots

LOSS_DIR = os.path.join(RESULT_DIR, "losses")
os.makedirs(LOSS_DIR, exist_ok=True)

loss_summary = {}
final_window = 100

for method, result in experiment_results.items():
    losses = result.get("losses", [])

    if len(losses) == 0:
        print(f"{method}: no training losses.")
        continue

    df = pd.DataFrame({
        "step": np.arange(1, len(losses) + 1),
        "loss": losses,
    })

    csv_path = os.path.join(LOSS_DIR, f"{method}_losses.csv")
    df.to_csv(csv_path, index=False)

    loss_summary[method] = {
        "num_steps": int(len(losses)),
        "final_loss": float(losses[-1]),
        "mean_loss": float(np.mean(losses)),
        f"last_{final_window}_mean_loss": float(np.mean(losses[-final_window:])),
        "csv_path": csv_path,
    }

    print(f"{method}: saved losses to {csv_path}")

summary_path = os.path.join(LOSS_DIR, "loss_summary.json")
with open(summary_path, "w") as f:
    json.dump(loss_summary, f, indent=2)

print("Saved loss summary to:", summary_path)

methods_with_losses = [m for m in ["lora", "dora", "full"] if m in loss_summary]

avg_losses = [
    loss_summary[m][f"last_{final_window}_mean_loss"]
    for m in methods_with_losses
]

plt.figure(figsize=(7, 5))
plt.bar(methods_with_losses, avg_losses)
plt.ylabel(f"Average Loss over Last {final_window} Steps")
plt.title("Final Average Training Loss")
plt.grid(axis="y", linestyle="--", alpha=0.5)

save_path = os.path.join(PLOT_DIR, "final_average_loss_bar.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved final average loss bar plot to:", save_path)

pd.DataFrame({
    "method": methods_with_losses,
    f"last_{final_window}_mean_loss": avg_losses,
})

In [ ]:
# Trainable Params
from diffusers import UNet2DConditionModel

def compute_param_stats_for_method(method):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    unet_tmp = UNet2DConditionModel.from_pretrained(
        MODEL_ID,
        subfolder="unet",
        torch_dtype=WEIGHT_DTYPE,
        local_files_only=True,
        use_safetensors=True,
    ).to(DEVICE)

    total_before = sum(p.numel() for p in unet_tmp.parameters())

    if method == "base":
        unet_tmp.requires_grad_(False)
        replace_count = 0
        trainable = 0
        total = total_before

    else:
        replace_count, trainable, total = inject_and_unfreeze(
            unet_tmp,
            ["to_q", "to_k", "to_v", "to_out.0"],
            RANK,
            ALPHA,
            DROPOUT,
            method,
        )

    stats = {
        "method": method,
        "replace_count": int(replace_count),
        "trainable_params": int(trainable),
        "total_params": int(total),
        "trainable_params_m": float(trainable / 1e6),
        "total_params_m": float(total / 1e6),
        "trainable_percent": float(100 * trainable / total if total > 0 else 0.0),
    }

    del unet_tmp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return stats


param_methods = [m for m in ["base", "lora", "dora", "full"] if m in experiment_results]

param_stats = []
for method in param_methods:
    print(f"Computing params for {method}...")
    stats = compute_param_stats_for_method(method)
    param_stats.append(stats)
    experiment_results[method].update(stats)

param_df = pd.DataFrame(param_stats)

param_csv_path = os.path.join(METRIC_DIR, "trainable_params_summary.csv")
param_json_path = os.path.join(METRIC_DIR, "trainable_params_summary.json")

param_df.to_csv(param_csv_path, index=False)
with open(param_json_path, "w") as f:
    json.dump(param_stats, f, indent=2)

print("Saved param summary to:")
print(param_csv_path)
print(param_json_path)

param_df

In [ ]:
plot_df = param_df[param_df["method"] != "base"].copy()

plt.figure(figsize=(7, 5))
plt.bar(plot_df["method"], plot_df["trainable_params_m"])
plt.ylabel("Trainable Parameters (M)")
plt.title("Trainable Parameter Count")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for idx, row in plot_df.iterrows():
    plt.text(
        row["method"],
        row["trainable_params_m"],
        f'{row["trainable_params_m"]:.2f}M\n{row["trainable_percent"]:.3f}%',
        ha="center",
        va="bottom",
        fontsize=9,
    )

save_path = os.path.join(PLOT_DIR, "trainable_params_bar.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved trainable params bar plot to:", save_path)

## CLIP Similarity Calculation

In [ ]:
!pip install -q transformers pillow

In [ ]:
import os
import gc
import torch
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel

# 0. Basic Checks
assert "experiment_results" in globals(), "experiment_results is not defined. Please run/load your experiment results first."

if "DEVICE" not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESULT_DIR = "/content/drive/My Drive/Colab_Notebooks/DoRA/results"
PLOT_DIR = os.path.join(RESULT_DIR, "plots")
METRIC_DIR = os.path.join(RESULT_DIR, "metrics")

os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(METRIC_DIR, exist_ok=True)

# 1. Load CLIP
clip_model_id = "openai/clip-vit-base-patch32"
clip_device = DEVICE

print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained(clip_model_id).to(clip_device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
clip_model.eval()

print("Loaded:", clip_model_id)
print("CLIP device:", clip_device)
print("CLIP class:", type(clip_model))

# 2. Style prompts
style_texts = [
    "traditional Chinese ink wash painting",
    "Chinese landscape painting on aged paper",
    "monochrome ink wash brush painting",
    "photograph",
    "oil painting",
    "anime illustration",
]

target_style = "traditional Chinese ink wash painting"

def extract_tensor_features(output):
    if torch.is_tensor(output):
        return output

    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output

    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0]

    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds

    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds

    raise TypeError(f"Cannot extract tensor features from output type: {type(output)}")

# 3. Compute style similarity
style_records = []

methods_to_eval = [m for m in ["base", "lora", "dora", "full"] if m in experiment_results]
print("Methods to evaluate:", methods_to_eval)

with torch.no_grad():
    text_inputs = clip_processor(
        text=style_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(clip_device)

    text_output = clip_model.get_text_features(**text_inputs)
    text_features = extract_tensor_features(text_output)
    text_features = F.normalize(text_features.float(), dim=-1)

    for method in methods_to_eval:
        images = experiment_results[method].get("images", [])
        print(f"{method}: {len(images)} images")

        for img_idx, img in enumerate(images):
            image_inputs = clip_processor(
                images=img,
                return_tensors="pt",
            ).to(clip_device)

            image_output = clip_model.get_image_features(**image_inputs)
            image_features = extract_tensor_features(image_output)
            image_features = F.normalize(image_features.float(), dim=-1)

            sims = (image_features @ text_features.T).squeeze(0).detach().cpu().numpy()

            record = {
                "method": method,
                "image_index": img_idx + 1,
            }

            for style, score in zip(style_texts, sims):
                record[style] = float(score)

            style_records.append(record)

style_df = pd.DataFrame(style_records)

style_csv_path = os.path.join(METRIC_DIR, "style_similarity_scores.csv")
style_df.to_csv(style_csv_path, index=False)

print("Saved style similarity scores to:", style_csv_path)
display(style_df.head())

# 4. Summarize
style_summary = (
    style_df
    .groupby("method")[target_style]
    .agg(["mean", "std", "count"])
    .reset_index()
)

style_summary_path = os.path.join(METRIC_DIR, "style_similarity_summary.csv")
style_summary.to_csv(style_summary_path, index=False)

print("Saved style summary to:", style_summary_path)
display(style_summary)

# 5. Plot
ordered_methods = [m for m in ["base", "lora", "dora", "full"] if m in style_summary["method"].values]
plot_summary = style_summary.set_index("method").loc[ordered_methods].reset_index()

plt.figure(figsize=(7, 5))
plt.bar(plot_summary["method"], plot_summary["mean"], yerr=plot_summary["std"])
plt.ylabel("CLIP Similarity")
plt.title(f'Style Similarity: "{target_style}"')
plt.grid(axis="y", linestyle="--", alpha=0.5)

for idx, row in plot_summary.iterrows():
    plt.text(
        idx,
        row["mean"],
        f'{row["mean"]:.3f}',
        ha="center",
        va="bottom",
        fontsize=9,
    )

save_path = os.path.join(PLOT_DIR, "style_similarity_bar.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved style similarity bar plot to:", save_path)

# 6. Cleanup optional
# del clip_model, clip_processor
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

In [ ]:
import pickle

pickle_path = os.path.join(RESULT_DIR, "experiment_results_with_metrics.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(experiment_results, f)

print("Saved updated experiment_results to:")
print(pickle_path)